# 01 Â· SmolVLA on the RTX 3060

Load the pretrained SmolVLA base policy and measure inference using the workbench's
two real camera views. **This is an inference benchmark, not a trained mouse-picking policy.**
There is currently no dataset or task-specific checkpoint for this robot.

**Kernel:** `Ramion Â· SmolVLA GPU`  
**Network:** the first load downloads public model weights; inference then runs locally.

The base model expects checkpoint-specific state/action units. Without a calibration
mapping, the benchmark deliberately uses zero state, masks the absent third camera,
and leaves outputs in model units. It cannot execute those outputs on the arm.


In [ ]:
import os
from pathlib import Path
from soarm_lab.notebooks.client import WorkbenchClient

REPO = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base / 'apps' / 'arm-workbench')
             if (p / 'packages' / 'soarm').is_dir()), None)
if REPO is None:
    raise RuntimeError('Open this notebook from the Ramion checkout.')
WORKSPACE = REPO / 'workspace' / 'notebooks'
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(REPO / 'workspace' / 'huggingface'))
client = WorkbenchClient(os.environ.get('RAMION_WORKBENCH_URL', 'http://100.85.56.14:8765'))
TASK = 'Move the stuffed toy mouse onto the left gray area.'


In [ ]:
from soarm_lab.notebooks.models import SmolVLAInference
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select the SmolVLA GPU kernel and verify CUDA before loading the model.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. Inspect the observation

Use an editable language instruction for the target area. Camera mapping is explicit:
`camera1` is overhead, `camera2` is wrist; the model's third slot is masked.


In [ ]:
TASK = 'Move the stuffed toy mouse onto the left gray area.'
snapshot = client.snapshot()
client.show(snapshot)


## 2. Load the base checkpoint

Set `LOAD_MODEL = True` to download/load weights. Keep other GPU model kernels closed
while loading. This cell claims no robot control.


In [ ]:
LOAD_MODEL = False
policy = None
if LOAD_MODEL:
    policy = SmolVLAInference(
        checkpoint='lerobot/smolvla_base',
        device='cuda',
        camera_map={
            'observation.images.camera1': 'overhead',
            'observation.images.camera2': 'wrist',
        },
        joint_mapping=None,
        trained_for_robot=False,
    )
    policy.load()
    print('Model loaded. No robot control has been claimed.')
else:
    print('Set LOAD_MODEL = True when ready to download/load the model.')


## 3. Run a short inference benchmark

The first result may include GPU warm-up overhead. All measurements use fresh images.
The output includes raw model actions so their shape and range can be inspected.
They are not degrees or gripper percentages unless a compatible checkpoint mapping says so.


In [ ]:
results = []
if policy is not None:
    for trial in range(3):
        observation = client.snapshot()
        result = policy.predict(observation.joints, observation.images, TASK, ages=observation.ages)
        results.append(result)
        print({'trial': trial + 1, 'elapsed_s': round(result.elapsed_s, 3),
               'executable': result.executable, 'note': result.note})
    print('Raw action shape:', results[-1].raw_actions.shape)
    print('First raw action rows:', results[-1].raw_actions[:3])
    print('GPU allocated MiB:', round(torch.cuda.memory_allocated() / 1024**2))
else:
    print('Load the model in the previous cell first.')


## What is needed before this route can move the mouse?

A task-specific or otherwise validated SO101 checkpoint must define its camera roles,
joint order, preprocessing statistics and action units. Supply the adapter with an
explicit mapping to the workbench's five joint angles in degrees and gripper percentage,
and mark the checkpoint as trained for this robot only after validation.

The default base-model path has no executable targets. Setting an arming flag cannot
turn this benchmark into a suitable manipulation policy. Demonstration collection and
fine-tuning are a separate future step; they are not silently performed here.

See [SmolVLA setup and training](https://huggingface.co/docs/lerobot/smolvla).


## Release GPU memory

Run this when switching models, or shut down this notebook's kernel in Jupyter.


In [ ]:
policy = None
results = []
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Model references released; shut down the kernel if GPU memory remains allocated.')
